![IITIS](pictures/logoIITISduze.png)

# Wyżarzanie równoległe

In [142]:
# Implementation of the algorithm
import random
import numpy as np
from typing import Optional

def project(x: float):
    if x == 0:
        return random.choice([-1, 1])
    else:
        return np.sign(x)


projector = np.vectorize(project)
clip = np.vectorize(lambda x: max(-1.0, min(1.0,x)))


def calculate_gradient(J: np.ndarray, h: np.ndarray, x: np.ndarray, state: np.ndarray, lambda_t: float) -> np.ndarray:
    return J @ state + h + lambda_t * x



def calculate_energy(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    return state @ J @ state.T + state @ h 


def parrarel_annealing(J, h, step_size: float, lambda_t_max: float, num_steps: int, schedule: Optional[np.ndarray] = None):
    n = len(h)
    x = np.zeros(n)
    momentum = 0

    if schedule is None:
        schedule = np.linspace(lambda_t_max, 1e-12, num=num_steps)

    for k in range(num_steps):
        x = clip(x)
        lambda_t = schedule[k]
        state = projector(x)
        gradient = calculate_gradient(J, h, x, state, lambda_t)
        momentum = momentum * 0.99 - step_size * gradient
        momentum = clip(momentum)
        x += momentum

    return projector(x), calculate_energy(J, h, projector(x))



    


In [141]:
n = 20

scaling_func = np.vectorize(lambda x: 2*x-1)  # przesunięcie wartości z (0, 1) do (-1, 1)
J = np.triu(scaling_func(np.random.rand(n, n)))  # losowa gęsta macierz górnotrójkątna
h = scaling_func(np.random.rand(n))  # losowy wektor




In [124]:
from dimod import BinaryQuadraticModel
from dwave.samplers import SimulatedAnnealingSampler

bqm_instance = BinaryQuadraticModel(h, J, vartype="SPIN")
sampler= SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_instance, num_reads=1)
print(sampleset)
print(parrarel_annealing(J, h, 10, 0.005, 10000))

   0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19    energy num_oc.
0 +1 -1 +1 +1 -1 +1 -1 +1 +1 +1 -1 +1 -1 +1 -1 +1 -1 -1 -1 +1 -43.41361       1
['SPIN', 1 rows, 1 samples, 20 variables]
(array([-1.,  1., -1.,  1., -1., -1.,  1., -1., -1., -1., -1., -1.,  1.,
       -1.,  1., -1., -1., -1.,  1.,  1.]), np.float64(-31.45542953394546))


In [153]:
# Działamy na większej instancji
# 216 spinów i geometryczna struktura QPU

import os
import pandas as pd
import numpy as np
from math import inf

instance_path = os.path.join("instances", "symulowane_wyżarzanie", "P4_CBFM-P.txt")
# W pliku jest poza samą instancję jeszcze najlepsza znaleziona energia E = -469


def read_instance(path: os.PathLike):
    df = pd.read_csv(path, sep=" ", header=None, comment="#", names=["i", "j", "value"])

    n = max(df[["i", "j"]].max())
    h = np.zeros(n)
    J = np.zeros((n, n))
    
    for row in df.itertuples():
        if row.i == row.j:
            h[row.i - 1] = row.value
        elif row.i > row.j:
            J[row.j - 1, row.i - 1] = row.value  # by zachować górnotrójkątność
        else:
            J[row.i - 1, row.j - 1] = row.value
            
    return h, J

    
h, J = read_instance(instance_path)
num_trajectories = 20

best_energy = inf
best_state = None

for _ in range(num_trajectories):
    state, energy = parrarel_annealing(J, h, 10, 0.0005, 10000)
    print(energy)
    if energy < best_energy:
        best_energy = energy
print(best_energy)

from dimod import BinaryQuadraticModel
from dwave.samplers import SimulatedAnnealingSampler

bqm_instance = BinaryQuadraticModel(h, J, vartype="SPIN")
sampler= SimulatedAnnealingSampler()
sampleset = sampler.sample(bqm_instance, num_reads=1)
print(sampleset)

-301.0
-285.0
-323.0
-309.0
-295.0
-311.0
-315.0
-319.0
-271.0
-325.0
-291.0
-309.0
-309.0
-291.0
-325.0
-329.0
-327.0
-287.0
-287.0
-293.0
-329.0
   0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 ... 215 energy num_oc.
0 +1 +1 +1 +1 -1 -1 -1 +1 +1 +1 +1 +1 -1 -1 -1 -1 +1 +1 ...  +1 -467.0       1
['SPIN', 1 rows, 1 samples, 216 variables]


In [ ]:
# Wyżarzanie równoległe wiele trajektorii

def parrarel_annealing_multiple_trajectores():
    ...
